# Exercise 3.2, Operator Averages Using the First Moment Formula

**Chapter 3: Haar Ensembles** &nbsp;|&nbsp; *Section 3.2: Weingarten Calculus*

---

## Background

The first-moment identity $\mathbb{E}_U[U A U^\dagger] = \tfrac{\mathrm{Tr}(A)}{D}\, I$ is the $k=1$ case of the Weingarten calculus and the starting point of Haar integration. It says that conjugation by a random unitary erases everything about an operator except its trace, which is Schur's lemma read through the averaging map and the operator form of complete depolarization. The notebook uses it to evaluate $\mathbb{E}_U[\mathrm{Tr}(U A U^\dagger B)]$, a contraction that recurs whenever one twirls an observable against a fixed reference.

## Exercise Statement

Let $A, B$ be arbitrary $D \times D$ complex matrices. Using the first moment formula, compute

$$
\mathbb{E}_U\bigl[\mathrm{Tr}(U A U^\dagger B)\bigr].
$$

## Solution

### Step 1: Apply the first moment formula

The trace is linear, so we can bring the Haar average inside:

$$
\mathbb{E}_U\bigl[\mathrm{Tr}(U A U^\dagger B)\bigr] = \mathrm{Tr}\!\left(\mathbb{E}_U[U A U^\dagger] \cdot B\right).
$$

By the first moment formula:

$$
\mathbb{E}_U[U A U^\dagger] = \frac{\mathrm{Tr}(A)}{D}\, I.
$$

### Step 2: Evaluate the trace

$$
\mathrm{Tr}\!\left(\frac{\mathrm{Tr}(A)}{D}\, I \cdot B\right) = \frac{\mathrm{Tr}(A)}{D}\, \mathrm{Tr}(B).
$$

### Result

$$
\boxed{\mathbb{E}_U\bigl[\mathrm{Tr}(U A U^\dagger B)\bigr] = \frac{\mathrm{Tr}(A)\,\mathrm{Tr}(B)}{D}.}
$$

**Interpretation:** The Haar average factorizes the product into individual traces, all correlations between $A$ and $B$ are erased. Only the "scalar content" (traces) of each operator survives the random rotation. This factorization is the trace-level manifestation of depolarization.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

D = sp.Symbol('D', positive=True, integer=True)

# Weingarten function for k=1: Wg(id, D) = 1/D
# The first moment formula:
# E[U_{ij} U*_{kl}] = (1/D) * delta_{il} * delta_{jk}
Wg_1 = sp.Rational(1, 1) / D
print(f'k=1 Weingarten function: Wg(id, D) = {Wg_1}')

# Consequence: E[U A U^dag] = Tr(A)/D * I
print(f'\nFirst moment: E[U A U^dag] = Tr(A)/D * I')

# Verify: taking trace of both sides
# Tr(E[U A U^dag]) = E[Tr(A)] = Tr(A)
# Tr(Tr(A)/D * I) = Tr(A)/D * D = Tr(A)  CHECK!
print(f'Trace consistency: Tr(Tr(A)/D * I) = Tr(A)/D * D = Tr(A)  PASS')

# k=2 Weingarten functions for reference
Wg2_id = 1 / (D**2 - 1)
Wg2_swap = -1 / (D * (D**2 - 1))
print(f'\nk=2 Weingarten functions:')
print(f'  Wg(id, D) = {Wg2_id}')
print(f'  Wg((12), D) = {Wg2_swap}')
print(f'  Sum: {sp.simplify(Wg2_id + Wg2_swap)} = 1/(D(D+1))')

---
## Numerical Verification

In [ ]:
import numpy as np
from scipy.stats import unitary_group

np.random.seed(42)

for D in [2, 3, 4, 8]:
    A = np.random.randn(D, D) + 1j*np.random.randn(D, D)
    B = np.random.randn(D, D) + 1j*np.random.randn(D, D)
    prediction = np.trace(A) * np.trace(B) / D
    
    mc_vals = []
    for _ in range(20000):
        U = unitary_group.rvs(D)
        mc_vals.append(np.trace(U @ A @ U.conj().T @ B))
    mc_est = np.mean(mc_vals)
    
    print(f"D={D}: E[Tr(UAU†B)] = {mc_est:.4f}  (pred Tr(A)Tr(B)/D = {prediction:.4f})")
    assert abs(mc_est - prediction) < 0.15*abs(prediction) + 0.1

print("\nFirst moment formula: trace factorization confirmed. ✓")

## Takeaway

Haar-averaging a conjugated operator keeps only its trace, so $\mathbb{E}_U[\mathrm{Tr}(U A U^\dagger B)] = \mathrm{Tr}(A)\,\mathrm{Tr}(B)/D$. All directional information in $A$ and $B$ is washed out, leaving a product of traces divided by the dimension.